RAG with LangChain

In [9]:
%pip install langchain-google-genai langchain langchain_core langchain_community langchain-chroma chromadb sentence-transformers dotenv pypdf langchain-huggingface -q
print("Dependencies installed succesfully!")

Note: you may need to restart the kernel to use updated packages.
Dependencies installed succesfully!


In [10]:
#loading the documents
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("air-rag.pdf")
docs = loader.load()

docs[0].metadata
print("Docs loaded.")

Ignoring wrong pointing object 85 0 (offset 0)
Ignoring wrong pointing object 89 0 (offset 0)
Ignoring wrong pointing object 133 0 (offset 0)
Ignoring wrong pointing object 178 0 (offset 0)
Ignoring wrong pointing object 346 0 (offset 0)
Ignoring wrong pointing object 347 0 (offset 0)


Docs loaded.


In [11]:
#split into chunks
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = splitter.split_documents(docs)
print("Length of chunks : ",len(chunks))
print("Docs split into chunks.")

Length of chunks :  104
Docs split into chunks.


In [12]:
#load embedding model form hugging face
from huggingface_hub import login
from dotenv import load_dotenv
import os

load_dotenv()

hf_token = os.getenv("HUGGINGFACE_TOKEN")
if not hf_token:
    raise ValueError("Missing Hugging Face token. Set HUGGINGFACE_TOKEN in your .env file.")

login(token = hf_token)


In [13]:
#creating embeddings
from langchain_huggingface import HuggingFaceEmbeddings

loading_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embedding model loaded")

embeddings = loading_model.embed_documents([c.page_content for c in chunks])
print("Total embeddings : ",len(embeddings))
print("Dimension of embeddings : ",len(embeddings[0]))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5709.55it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded
Total embeddings :  104
Dimension of embeddings :  384


In [14]:
#storing in vector db
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents = chunks,
    collection_name = "docs_collection",
    embedding = loading_model,
    persist_directory = "./chroma_db"
)
print("Vector Store created.")

Vector Store created.


In [15]:
#retrieving from vector db
retriever = vector_store.as_retriever(
    search_type = 'similarity',
    search_kwargs = {'k' : 3}
)
print("Retriver loaded.")
results = retriever.invoke("What is the title of the paper?")
print(results)

Retriver loaded.
[Document(id='c1810674-cd8b-4d85-b9ff-7926ac3afb6a', metadata={'aapl:keywords': "['LLM,RAG']", 'title': 'Adaptive iterative retrieval for enhanced retrieval-augmented generation', 'source': 'air-rag.pdf', 'page': 10, 'total_pages': 14, 'author': 'Wenhan Han', 'creator': 'Elsevier', 'moddate': "D:20260131174331Z00'00'", 'subject': 'Neurocomputing, 666 (2026) 132272. doi:10.1016/j.neucom.2025.132272', 'creationdate': "D:20260131174331Z00'00'", 'keywords': 'LLM,RAG', 'page_label': '11', 'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext'}, page_content='Iter-2 Sentences\ns_1\x01 Kate Millett Katherine Murray Millett (September 14, 1934 – September 6, 2017) was an\nAmerican feminist writer, educator, artist, and activist.\x01\n \ns_2 She attended Oxford University and was the first American woman to be awarded a degree\nwith first-class honors after studying at St Hilda’s College, Oxford.\x01\n \ns_3 She has been described as “a seminal influence on second-wav

In [18]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context given below.
If you don't know the answer, say you don't know.
                                          
Context : {context}
Question : {question}
""")

In [19]:
from dotenv import load_dotenv
import os

load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model = "gemini-flash-latest",
    temperature = 0
)

print("LLM loaded.")

LLM loaded.
